In [0]:
%sql
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_sales_product_performance AS
SELECT
    month_key,
    product_category,
    product_type,
    SUM(revenue)          AS total_revenue,
    SUM(transaction_qty)  AS total_units_sold,
    COUNT(transaction_id) AS num_transactions,
    ROUND(AVG(unit_price), 2) AS avg_unit_price,
    hour,
    CASE
        WHEN hour BETWEEN 6  AND 11 THEN 'Morning (6-11am)'
        WHEN hour BETWEEN 12 AND 14 THEN 'Lunch (12-2pm)'
        WHEN hour BETWEEN 15 AND 17 THEN 'Afternoon (3-5pm)'
        WHEN hour BETWEEN 18 AND 21 THEN 'Evening (6-9pm)'
        ELSE 'Other'
    END AS time_of_day
FROM cash_flow_project.cash_flow_gold.fact_sales
GROUP BY month_key, product_category, product_type, hour
ORDER BY month_key, total_revenue DESC

In [0]:
%sql
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_payroll_analysis AS
SELECT
    p.month_key,
    p.employee_name,
    p.role,
    p.type                      AS pay_type,
    p.total_paid,
    m.cash_revenue,
    ROUND(
        CASE WHEN m.cash_revenue > 0
             THEN p.total_paid / m.cash_revenue * 100
             ELSE 0 END, 1)     AS pct_of_monthly_revenue,
    m.shortage_flag
FROM cash_flow_project.cash_flow_gold.fact_payroll_monthly p
LEFT JOIN cash_flow_project.cash_flow_gold.fact_monthly_cashflow m
    ON p.month_key = m.month_key
ORDER BY p.month_key, p.total_paid DESC

In [0]:
%sql
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_shortage_detection AS
SELECT
    month_key,
    shortage_flag,
    shortage_severity,
    ROUND(cash_revenue, 2)                        AS monthly_income,
    ROUND(total_expenses, 2)                      AS monthly_expenses,
    ROUND(net_cash_flow, 2)                       AS net_cash_flow,
    ROUND(month_end_balance, 2)                   AS closing_balance,
    ROUND(avg_3m_revenue, 2)                      AS avg_3m_income,
    ROUND(avg_3m_expenses, 2)                     AS avg_3m_expenses,
    ROUND(
        CASE WHEN avg_3m_revenue > 0
             THEN (cash_revenue - avg_3m_revenue) / avg_3m_revenue * 100
             ELSE 0 END, 1)                       AS income_vs_avg_pct,
    ROUND(
        CASE WHEN avg_3m_expenses > 0
             THEN (total_expenses - avg_3m_expenses) / avg_3m_expenses * 100
             ELSE 0 END, 1)                       AS expense_vs_avg_pct,
    needs_ai_suggestion,
    -- Human-readable explanation of why the flag fired
    CASE shortage_flag
        WHEN 'CRITICAL'      THEN 'Net cash flow is negative — spent more than earned'
        WHEN 'EXPENSE_SPIKE' THEN 'Expenses are 20%+ above 3-month average'
        WHEN 'INCOME_DROP'   THEN 'Revenue dropped 20%+ below 3-month average'
        WHEN 'WARNING'       THEN 'Balance below 1.5x monthly expenses — low buffer'
        ELSE 'No issues detected'
    END                                           AS flag_explanation
FROM cash_flow_project.cash_flow_gold.fact_monthly_cashflow
ORDER BY shortage_severity DESC, month_key